### Source Tables:
- _exponent._bronze_allscripts_scm_prod_01.dbo_cv3clientvisit (115M visits)
- _exponent._bronze_allscripts_scm_prod_01.dbo_cv3client (patient demographics)

### To Do:
- Map TypeCode to OMOP visit_concept_id using domain_source_to_concept
- Map CareLevelCode if needed for additional context
- Link to provider_id once provider table is populated
- Link to care_site_id using CurrentLocationGUID once care_site table is populated

### Notes:
- PERSON must run before VISIT_OCCURRENCE
- cv3clientvisit.GUID is the visit identifier (DECIMAL type)
- cv3clientvisit.ClientGUID links to cv3client.GUID
- Visit dates: AdmitDtm, DischargeDtm
- Visit type: TypeCode, CareLevelCode
- Visit status: VisitStatus (CLS=Closed, etc.)

In [0]:
%sql
-- Reset Gold
TRUNCATE TABLE _exponent.omop_scm.visit_occurrence;


In [0]:
%sql
-- Reset Silver
DELETE FROM _exponent.omop_silver.visit_occurrence
WHERE source_system = 'allscripts_scm';


In [0]:
%sql
-- Reset Mapping
DELETE FROM _exponent.omop_mapping.source_to_visit_occurrence
WHERE source_system = 'allscripts_scm';


# Transformation

In [0]:
source = 'allscripts_scm'

In [0]:
silver_visit_df = spark.sql(f'''
SELECT 
  source_to_person.person_id,
  COALESCE(visit_concept.omop_concept_id, 9202) AS visit_concept_id,  -- Default to 9202 = Outpatient Visit
  CAST(v.AdmitDtm AS DATE) AS visit_start_date,
  v.AdmitDtm AS visit_start_datetime,
  CAST(COALESCE(v.DischargeDtm, v.AdmitDtm) AS DATE) AS visit_end_date,
  COALESCE(v.DischargeDtm, v.AdmitDtm) AS visit_end_datetime,
  32817 AS visit_type_concept_id,  -- 32817 = EHR
  NULL AS provider_id,  -- TODO: Map to provider once provider table is populated
  NULL AS care_site_id,  -- TODO: Map CurrentLocationGUID to care_site_id
  CONCAT_WS(CHR(31), '{source}', 'dbo_cv3clientvisit', 'GUID', CAST(v.GUID AS STRING)) AS visit_source_value,
  COALESCE(visit_concept.omop_concept_id, 0) AS visit_source_concept_id,
  NULL AS admitted_from_concept_id,
  NULL AS admitted_from_source_value,
  NULL AS discharged_to_concept_id,
  v.DischargeDisposition AS discharged_to_source_value,
  NULL AS preceding_visit_occurrence_id,
  '{source}' AS source_system
FROM `_exponent`.`_bronze_allscripts_scm_prod_01`.`dbo_cv3clientvisit` v
INNER JOIN _exponent.omop_mapping.source_to_person
  ON CONCAT_WS(CHR(31), '{source}', 'cv3client', 'GUID', CAST(v.ClientGUID AS STRING)) = source_to_person.person_source_value
  AND source_to_person.active_flag = TRUE
LEFT JOIN _exponent.omop_mapping.domain_source_to_concept visit_concept
  ON visit_concept.source_id = v.TypeCode
  AND visit_concept.domain_id = 'Visit'
  AND visit_concept.source_system = '{source}'
WHERE v.GUID IS NOT NULL
  AND v.ClientGUID IS NOT NULL
  AND v.AdmitDtm IS NOT NULL
  AND v.AdmitDtm >= TIMESTAMP('1900-01-01')
  AND v.AdmitDtm <= CURRENT_TIMESTAMP()
  AND COALESCE(v.DischargeDtm, v.AdmitDtm) >= TIMESTAMP('1900-01-01')
  AND COALESCE(v.DischargeDtm, v.AdmitDtm) <= CURRENT_TIMESTAMP()
''')

display(silver_visit_df)
silver_visit_df.createOrReplaceTempView("silver_visit")

In [0]:
%sql
MERGE INTO _exponent.omop_silver.visit_occurrence AS t
USING silver_visit AS s
ON t.visit_source_value = s.visit_source_value

WHEN MATCHED AND (
     NOT (t.person_id <=> s.person_id)
  OR NOT (t.visit_concept_id <=> s.visit_concept_id)
  OR NOT (t.visit_start_date <=> s.visit_start_date)
  OR NOT (t.visit_start_datetime <=> s.visit_start_datetime)
  OR NOT (t.visit_end_date <=> s.visit_end_date)
  OR NOT (t.visit_end_datetime <=> s.visit_end_datetime)
  OR NOT (t.visit_type_concept_id <=> s.visit_type_concept_id)
  OR NOT (t.provider_id <=> s.provider_id)
  OR NOT (t.care_site_id <=> s.care_site_id)
  OR NOT (t.visit_source_concept_id <=> s.visit_source_concept_id)
  OR NOT (t.admitted_from_concept_id <=> s.admitted_from_concept_id)
  OR NOT (t.admitted_from_source_value <=> s.admitted_from_source_value)
  OR NOT (t.discharged_to_concept_id <=> s.discharged_to_concept_id)
  OR NOT (t.discharged_to_source_value <=> s.discharged_to_source_value)
  OR NOT (t.preceding_visit_occurrence_id <=> s.preceding_visit_occurrence_id)
  OR NOT (t.source_system <=> s.source_system)
)
THEN UPDATE SET
  t.person_id                       = s.person_id,
  t.visit_concept_id                = s.visit_concept_id,
  t.visit_start_date                = s.visit_start_date,
  t.visit_start_datetime            = s.visit_start_datetime,
  t.visit_end_date                  = s.visit_end_date,
  t.visit_end_datetime              = s.visit_end_datetime,
  t.visit_type_concept_id           = s.visit_type_concept_id,
  t.provider_id                     = s.provider_id,
  t.care_site_id                    = s.care_site_id,
  t.visit_source_concept_id         = s.visit_source_concept_id,
  t.admitted_from_concept_id        = s.admitted_from_concept_id,
  t.admitted_from_source_value      = s.admitted_from_source_value,
  t.discharged_to_concept_id        = s.discharged_to_concept_id,
  t.discharged_to_source_value      = s.discharged_to_source_value,
  t.preceding_visit_occurrence_id   = s.preceding_visit_occurrence_id,
  t.source_system                   = s.source_system

WHEN NOT MATCHED THEN INSERT (
  person_id,
  visit_concept_id,
  visit_start_date,
  visit_start_datetime,
  visit_end_date,
  visit_end_datetime,
  visit_type_concept_id,
  provider_id,
  care_site_id,
  visit_source_value,
  visit_source_concept_id,
  admitted_from_concept_id,
  admitted_from_source_value,
  discharged_to_concept_id,
  discharged_to_source_value,
  preceding_visit_occurrence_id,
  source_system
)
VALUES (
  s.person_id,
  s.visit_concept_id,
  s.visit_start_date,
  s.visit_start_datetime,
  s.visit_end_date,
  s.visit_end_datetime,
  s.visit_type_concept_id,
  s.provider_id,
  s.care_site_id,
  s.visit_source_value,
  s.visit_source_concept_id,
  s.admitted_from_concept_id,
  s.admitted_from_source_value,
  s.discharged_to_concept_id,
  s.discharged_to_source_value,
  s.preceding_visit_occurrence_id,
  s.source_system
);

In [0]:
%sql
INSERT INTO _exponent.omop_mapping.source_to_visit_occurrence (
    source_system,
    visit_occurrence_source_value,
    person_id,
    active_flag,
    created_tsp,
    last_mod_tsp,
    merge_id,
    merge_reason
)
SELECT
    s.source_system,
    s.visit_source_value AS visit_occurrence_source_value,
    s.person_id,
    TRUE AS active_flag,
    current_timestamp() AS created_tsp,
    current_timestamp() AS last_mod_tsp,
    NULL AS merge_id,
    NULL AS merge_reason
FROM (
    SELECT DISTINCT 
        source_system, 
        visit_source_value,
        person_id
    FROM _exponent.omop_silver.visit_occurrence
    WHERE source_system = 'allscripts_scm'
) s
LEFT ANTI JOIN _exponent.omop_mapping.source_to_visit_occurrence x
  ON s.visit_source_value = x.visit_occurrence_source_value;

In [0]:
%sql
MERGE INTO _exponent.omop_scm.visit_occurrence AS gold_visit
USING (
  SELECT
    source_to_visit_occurrence.visit_occurrence_id,
    s.person_id,
    s.visit_concept_id,
    s.visit_start_date,
    s.visit_start_datetime,
    s.visit_end_date,
    s.visit_end_datetime,
    s.visit_type_concept_id,
    s.provider_id,
    s.care_site_id,
    s.visit_source_value,
    s.visit_source_concept_id,
    s.admitted_from_concept_id,
    s.admitted_from_source_value,
    s.discharged_to_concept_id,
    s.discharged_to_source_value,
    s.preceding_visit_occurrence_id
  FROM _exponent.omop_silver.visit_occurrence s
  JOIN _exponent.omop_mapping.source_to_visit_occurrence
    ON source_to_visit_occurrence.visit_occurrence_source_value = s.visit_source_value
   AND source_to_visit_occurrence.active_flag = TRUE
  WHERE s.source_system = 'allscripts_scm'
) AS src
ON gold_visit.visit_occurrence_id = src.visit_occurrence_id

WHEN MATCHED AND NOT (
     gold_visit.person_id                     <=> src.person_id
 AND gold_visit.visit_concept_id              <=> src.visit_concept_id
 AND gold_visit.visit_start_date              <=> src.visit_start_date
 AND gold_visit.visit_start_datetime          <=> src.visit_start_datetime
 AND gold_visit.visit_end_date                <=> src.visit_end_date
 AND gold_visit.visit_end_datetime            <=> src.visit_end_datetime
 AND gold_visit.visit_type_concept_id         <=> src.visit_type_concept_id
 AND gold_visit.provider_id                   <=> src.provider_id
 AND gold_visit.care_site_id                  <=> src.care_site_id
 AND gold_visit.visit_source_value            <=> src.visit_source_value
 AND gold_visit.visit_source_concept_id       <=> src.visit_source_concept_id
 AND gold_visit.admitted_from_concept_id      <=> src.admitted_from_concept_id
 AND gold_visit.admitted_from_source_value    <=> src.admitted_from_source_value
 AND gold_visit.discharged_to_concept_id      <=> src.discharged_to_concept_id
 AND gold_visit.discharged_to_source_value    <=> src.discharged_to_source_value
 AND gold_visit.preceding_visit_occurrence_id <=> src.preceding_visit_occurrence_id
)
THEN UPDATE SET
  gold_visit.person_id                     = src.person_id,
  gold_visit.visit_concept_id              = src.visit_concept_id,
  gold_visit.visit_start_date              = src.visit_start_date,
  gold_visit.visit_start_datetime          = src.visit_start_datetime,
  gold_visit.visit_end_date                = src.visit_end_date,
  gold_visit.visit_end_datetime            = src.visit_end_datetime,
  gold_visit.visit_type_concept_id         = src.visit_type_concept_id,
  gold_visit.provider_id                   = src.provider_id,
  gold_visit.care_site_id                  = src.care_site_id,
  gold_visit.visit_source_value            = src.visit_source_value,
  gold_visit.visit_source_concept_id       = src.visit_source_concept_id,
  gold_visit.admitted_from_concept_id      = src.admitted_from_concept_id,
  gold_visit.admitted_from_source_value    = src.admitted_from_source_value,
  gold_visit.discharged_to_concept_id      = src.discharged_to_concept_id,
  gold_visit.discharged_to_source_value    = src.discharged_to_source_value,
  gold_visit.preceding_visit_occurrence_id = src.preceding_visit_occurrence_id

WHEN NOT MATCHED THEN INSERT (
  visit_occurrence_id,
  person_id,
  visit_concept_id,
  visit_start_date,
  visit_start_datetime,
  visit_end_date,
  visit_end_datetime,
  visit_type_concept_id,
  provider_id,
  care_site_id,
  visit_source_value,
  visit_source_concept_id,
  admitted_from_concept_id,
  admitted_from_source_value,
  discharged_to_concept_id,
  discharged_to_source_value,
  preceding_visit_occurrence_id
)
VALUES (
  src.visit_occurrence_id,
  src.person_id,
  src.visit_concept_id,
  src.visit_start_date,
  src.visit_start_datetime,
  src.visit_end_date,
  src.visit_end_datetime,
  src.visit_type_concept_id,
  src.provider_id,
  src.care_site_id,
  src.visit_source_value,
  src.visit_source_concept_id,
  src.admitted_from_concept_id,
  src.admitted_from_source_value,
  src.discharged_to_concept_id,
  src.discharged_to_source_value,
  src.preceding_visit_occurrence_id
);